# 09 · Storytelling outputs

Este notebook consolida saídas visuais finais (`final_*`) para a apresentação.
Ele resume Q1–Q5 e explicita as limitações metodológicas que não podem ser omitidas.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:,.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.figsize': (12, 5), 'figure.dpi': 120})


def locate_project_root() -> Path:
    start = Path.cwd().resolve()
    for base in [start, *start.parents]:
        if (base / 'data' / 'derived').exists():
            return base
    raise FileNotFoundError('Could not locate the repo root from the current working directory.')


PROJECT_ROOT = locate_project_root()
DERIVED = PROJECT_ROOT / 'data' / 'derived'
FIGURES = DERIVED / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
WEATHER_ORDER = ['Clear', 'Light Rain', 'Moderate Rain', 'Heavy Rain / Storm']


def savefig(name: str) -> Path:
    path = FIGURES / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    print(f'Saved figure: {path}')
    return path


In [ ]:
weather = pd.read_parquet(DERIVED / 'weather_hourly.parquet')
ticket = pd.read_parquet(DERIVED / 'ticket_hourly_route_profile.parquet')
integrated = pd.read_parquet(DERIVED / 'integrated_route_hour.parquet')
quality = pd.read_csv(DERIVED / 'mobility_day_quality_flags.csv', parse_dates=['date'])
trip_coverage = pd.read_csv(DERIVED / 'trip_base_coverage.csv')
crosswalk = pd.read_csv(DERIVED / 'route_crosswalk.csv')

weather_clean = weather.loc[~weather['weather_observation_missing']].copy()
daily_ticket = ticket.groupby('date', as_index=False).agg(boardings=('boardings', 'sum'))
daily_weather = weather_clean.groupby('date', as_index=False).agg(rain_mm=('rain_mm', 'sum'))
daily_weather['weather_cat'] = pd.Categorical(
    pd.cut(daily_weather['rain_mm'], bins=[-np.inf, 1, 10, 25, np.inf], labels=WEATHER_ORDER, right=False),
    categories=WEATHER_ORDER,
    ordered=True,
)
daily = daily_ticket.merge(daily_weather, on='date', how='inner')

print('=== Final storytelling context ===')
print('Ticket days:', pd.to_datetime(ticket['date']).dt.date.nunique())
print('Integrated days:', pd.to_datetime(integrated['date']).dt.date.nunique())
print('Flagged mobility days:', int(quality['is_partial_day'].sum()))
print('Unmatched trip bases:', int((~trip_coverage['in_gtfs']).sum()))
print('Unresolved ticket routes excluded:', int(crosswalk['manual_review'].sum()))


In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()
ax1.bar(pd.to_datetime(daily['date']), daily['boardings'], color='#4C78A8', alpha=0.85)
ax2.plot(pd.to_datetime(daily['date']), daily['rain_mm'], color='#D62728', marker='o', linewidth=2)
ax1.set_title('Final Q1 · Demand vs rainfall in March/2026')
ax1.set_ylabel('Boardings')
ax2.set_ylabel('Daily rainfall (mm)')
ax1.tick_params(axis='x', rotation=45)
savefig('final_q1_demand_weather.png')
plt.show()

profile_weather = (
    ticket.groupby(['card_label', 'date', 'hour'], observed=True, as_index=False)
    .agg(boardings=('boardings', 'sum'))
    .merge(weather_clean[['date', 'hour', 'weather_cat']], on=['date', 'hour'], how='inner')
)
heatmap = (
    profile_weather.groupby(['card_label', 'weather_cat'], observed=True)['boardings']
    .mean()
    .unstack(fill_value=np.nan)
    .reindex(columns=WEATHER_ORDER)
)
normalized = heatmap.div(heatmap['Clear'], axis=0) - 1
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.heatmap(normalized * 100, cmap='coolwarm', center=0, annot=True, fmt='.1f', ax=ax)
ax.set_title('Final Q2 · Relative change vs clear weather by passenger profile (%)')
savefig('final_q2_profile_heterogeneity.png')
plt.show()


In [ ]:
service = integrated.dropna(subset=['observed_trip_count']).copy()
service = service.loc[service['coverage_flag'].fillna('ok') != 'partial_day'].copy()
service['weather_bucket'] = pd.Categorical(
    np.where(service['rain_mm'].fillna(0) >= 10, 'Adverse (>=10 mm)', np.where(service['rain_mm'].fillna(0) > 0, 'Rain (<10 mm)', 'Clear')),
    categories=['Clear', 'Rain (<10 mm)', 'Adverse (>=10 mm)'],
    ordered=True,
)

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=service, x='weather_bucket', y='service_gap_index', ax=ax)
ax.set_title('Final Q3 · Service gap index under different weather buckets')
ax.tick_params(axis='x', rotation=15)
savefig('final_q3_service_degradation.png')
plt.show()

route_weather = (
    service.assign(rain_flag=np.where(service['rain_mm'].fillna(0) > 0, 'Rain', 'Clear'))
    .groupby(['route_norm', 'operator', 'rain_flag'], observed=True)
    .agg(boardings=('boardings', 'mean'), headway_p50=('headway_p50', 'mean'))
    .reset_index()
)
clear = route_weather.loc[route_weather['rain_flag'] == 'Clear'].copy()
rain = route_weather.loc[route_weather['rain_flag'] == 'Rain'].copy()
route_delta = clear.merge(rain, on=['route_norm', 'operator'], suffixes=('_clear', '_rain'))
route_delta['resilience_score'] = -(route_delta['boardings_rain'] / route_delta['boardings_clear'] - 1) - (route_delta['headway_p50_rain'] / route_delta['headway_p50_clear'] - 1)
top_vulnerable = route_delta.sort_values('resilience_score', ascending=False).head(10)
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=top_vulnerable, x='route_norm', y='resilience_score', hue='operator', ax=ax)
ax.set_title('Final Q4 · Most vulnerable routes in rainy conditions')
ax.tick_params(axis='x', rotation=45)
savefig('final_q4_route_resilience.png')
plt.show()


In [ ]:
model_df = integrated.loc[integrated['coverage_flag'].fillna('ok') != 'partial_day'].copy()
model_df = model_df.dropna(subset=['boardings', 'rain_mm', 'headway_p50', 'speed_p50', 'service_gap_index', 'hour', 'day_of_week', 'route_norm']).copy()
base_model = smf.glm(
    'boardings ~ rain_mm + C(route_norm) + C(hour) + C(day_of_week)',
    data=model_df,
    family=sm.families.Poisson(),
).fit(cov_type='HC1')
service_model = smf.glm(
    'boardings ~ rain_mm + headway_p50 + speed_p50 + service_gap_index + C(route_norm) + C(hour) + C(day_of_week)',
    data=model_df,
    family=sm.families.Poisson(),
).fit(cov_type='HC1')
interaction_model = smf.glm(
    'boardings ~ rain_mm + headway_p50 + speed_p50 + service_gap_index + rain_mm:headway_p50 + rain_mm:speed_p50 + rain_mm:service_gap_index + C(route_norm) + C(hour) + C(day_of_week)',
    data=model_df,
    family=sm.families.Poisson(),
).fit(cov_type='HC1')
nested = pd.DataFrame(
    {
        'model': ['base', 'service', 'interaction'],
        'rain_pct_effect': [100 * (np.exp(base_model.params['rain_mm']) - 1), 100 * (np.exp(service_model.params['rain_mm']) - 1), 100 * (np.exp(interaction_model.params['rain_mm']) - 1)],
        'aic': [base_model.aic, service_model.aic, interaction_model.aic],
    }
)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=nested, x='model', y='rain_pct_effect', ax=ax)
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Final Q5 · Rain effect across nested demand models')
ax.set_ylabel('Percent effect of 1 mm rain')
savefig('final_q5_joint_effects.png')
plt.show()


In [ ]:
limitations = pd.DataFrame(
    {
        'limitation': [
            'Q3–Q5 rely on the integrated 19-day window',
            'Heavy Rain / Storm is absent from the observed month',
            'Five mobility trip bases remain unmatched to GTFS',
            'Five ticket routes remain unresolved and were excluded from integrated joins',
            'Four mobility days were flagged as partial and tested via sensitivity analyses',
        ]
    }
)
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.axis('off')
table = ax.table(
    cellText=[[idx + 1, value] for idx, value in enumerate(limitations['limitation'])],
    colLabels=['#', 'Methodological limitation'],
    loc='center',
    cellLoc='left',
    colLoc='left',
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.6)
ax.set_title('Final limitations that must remain in the presentation')
savefig('final_limitations_table.png')
plt.show()
print(limitations.to_string(index=False))


## Mensagens seguras para a apresentação

1. **Q1:** há base suficiente para discutir demanda em tempo firme, chuva leve e chuva moderada.
2. **Q2:** a resposta à chuva não precisa ser homogênea entre perfis de passageiro; a heterogeneidade foi testada explicitamente.
3. **Q3–Q5:** os resultados operacionais e conjuntos devem ser comunicados como achados da **janela integrada de 19 dias**, não do mês inteiro.
4. **Eventos severos:** como `Heavy Rain / Storm` não aparece nas observações, não se deve extrapolar para tempestades fortes.
